# Day 39: Build, Debug and Test

**Today's question:** how do you build a whole application without ending up with 300 lines that "almost work"?

Short answer: **small steps**, each one run and tested before the next. Today's examples come from a sample
**Budget Tracker** module (capstone option a).

## 1. The build loop

```
pick ONE small step ──▶ write it ──▶ run it ──▶ test it ──▶ checkpoint ──┐
        ▲                                                               │
        └───────────────────────────────────────────────────────────────┘
```

A good step is **commit-sized**: one function, one method or one menu option; about 10–20 minutes; and it ends
with the app running and every test passing. Then make a **checkpoint**: a `git commit` if you use git,
otherwise a copy of your project folder.

Example steps for the Day 38 Recipe Box:

| # | Step | Done when |
|---|------|-----------|
| 1 | `Recipe.__init__` + `__str__` + 2 tests | `uv run pytest -q` → 2 passed |
| 2 | `Recipe.add_ingredient` refuses empty/duplicate + 2 tests | 4 passed |
| 3 | `save_recipes` / `load_recipes` + save-then-load test | restart keeps a recipe |
| 4 | Menu 1 "list recipes" uses `load_recipes` | the menu shows real data |
| 5 | Menu 2 "add a recipe" with `ask_text`/`ask_int` | bad input asks again |

## 2. A module with bugs

A classmate wrote these functions for their budget tracker. The app runs without any error message…

In [ ]:
def balance(transactions):
    """Return total income minus total expenses."""
    total = 0
    for item in transactions:
        if item["kind"] == "income":
            total -= item["amount"]
        else:
            total += item["amount"]
    return total


def spending_by_category(transactions):
    """Return {category: total spent} for expenses only."""
    totals = {}
    for item in transactions:
        if item["kind"] == "expense":
            totals[item["category"]] = item["amount"]
    return totals


def largest_expense(transactions):
    """Return the expense with the biggest amount."""
    biggest = transactions[0]
    for item in transactions:
        if item["kind"] == "expense" and item["amount"] > biggest["amount"]:
            biggest = item
        return biggest

Tiny data whose answers we can work out **in our heads**: income 1000, expenses 200 + 50 + 30.
🤔 **Predict:** what *should* the balance be? What will it be?

In [ ]:
sample = [
    {"kind": "income", "category": "salary", "amount": 1000},
    {"kind": "expense", "category": "rent", "amount": 200},
    {"kind": "expense", "category": "food", "amount": 50},
    {"kind": "expense", "category": "food", "amount": 30},
]
print(balance(sample))
print(spending_by_category(sample))
print(largest_expense(sample))

No error message, but **all three answers are wrong**. Bugs that do not crash are the dangerous ones: only a
test that knows the right answer catches them.

## 3. Tests catch the bugs

Each test states the right answer for the tiny data. `run_tests` is a mini-pytest: it runs each test and
reports PASS or FAIL instead of stopping at the first failure.

In [ ]:
def test_balance():
    assert balance(sample) == 720


def test_spending_by_category():
    assert spending_by_category(sample) == {"rent": 200, "food": 80}


def test_largest_expense():
    assert largest_expense(sample)["category"] == "rent"


def run_tests(tests):
    """Run (name, test_function) pairs and print PASS or FAIL for each."""
    for name, test in tests:
        try:
            test()
            print("PASS", name)
        except AssertionError:
            print("FAIL", name)


all_tests = [("balance", test_balance), ("spending_by_category", test_spending_by_category),
             ("largest_expense", test_largest_expense)]
run_tests(all_tests)

🤔 For each FAIL, find the bug **by reading the function**. Hints: a sign, `=` vs `+=`, and indentation.

Then a crash on the very first run of the app, when there are **no** transactions yet:

In [ ]:
def average_expense(transactions):
    """Return the average amount of the expenses."""
    expenses = [item["amount"] for item in transactions if item["kind"] == "expense"]
    return sum(expenses) / len(expenses)

In [ ]:
average_expense([])

Read the traceback **from the bottom up** (Day 28): the last line says *what* (`ZeroDivisionError`), the line
above it says *where* (`return sum(expenses) / len(expenses)`). Ask: "which value made it fail?" → `len(expenses)`
is 0. The fix is a **decision**: what should the average of no expenses be? Here we return `0.0`.

In [ ]:
def balance(transactions):
    """Return total income minus total expenses."""
    total = 0
    for item in transactions:
        if item["kind"] == "income":
            total += item["amount"]
        else:
            total -= item["amount"]
    return total


def spending_by_category(transactions):
    """Return {category: total spent} for expenses only."""
    totals = {}
    for item in transactions:
        if item["kind"] == "expense":
            totals[item["category"]] = totals.get(item["category"], 0) + item["amount"]
    return totals


def largest_expense(transactions):
    """Return the expense with the biggest amount, or None if there are no expenses."""
    biggest = None
    for item in transactions:
        if item["kind"] == "expense":
            if biggest is None or item["amount"] > biggest["amount"]:
                biggest = item
    return biggest


def average_expense(transactions):
    """Return the average amount of the expenses (0.0 when there are none)."""
    expenses = [item["amount"] for item in transactions if item["kind"] == "expense"]
    if not expenses:
        return 0.0
    return sum(expenses) / len(expenses)

In [ ]:
def test_empty_data():
    assert balance([]) == 0
    assert largest_expense([]) is None
    assert average_expense([]) == 0.0


all_tests.append(("empty data", test_empty_data))
run_tests(all_tests)

Notice the fix for `largest_expense`: the old version started from `transactions[0]`, which could be the
*income*. Every bug got a test, so it cannot come back unnoticed. In your project these functions go in
`test_budget.py` and `uv run pytest -q` does what `run_tests` does here.

**Debugging moves, in order:** read the traceback bottom-up → print the variables just before the failing line →
cut the input down to the smallest example that still fails → explain the code line by line to a classmate
(or a rubber duck) → use the VS Code debugger with a breakpoint (Day 28).

## 4. Refactoring a long function

**Refactoring** = changing the structure **without changing what the code does**. This works, but it does
four jobs in one function:

In [ ]:
def monthly_report_long(transactions, month):
    """Return a text report for one month (YYYY-MM)."""
    income = 0
    spent = 0
    categories = {}
    for item in transactions:
        if item["date"][:7] == month:
            if item["kind"] == "income":
                income += item["amount"]
            else:
                spent += item["amount"]
                categories[item["category"]] = categories.get(item["category"], 0) + item["amount"]
    lines = [f"Report for {month}", f"Income: {income:.2f}", f"Spent:  {spent:.2f}"]
    for category, total in categories.items():
        lines.append(f"  {category:<10}{total:>8.2f}")
    lines.append(f"Balance: {income - spent:.2f}")
    return "\n".join(lines)


history = [
    {"date": "2026-11-28", "kind": "expense", "category": "food", "amount": 40},
    {"date": "2026-12-01", "kind": "income", "category": "salary", "amount": 1000},
    {"date": "2026-12-01", "kind": "expense", "category": "rent", "amount": 200},
    {"date": "2026-12-03", "kind": "expense", "category": "food", "amount": 50},
]
before = monthly_report_long(history, "2026-12")
print(before)

Step 1 of every refactoring: **save the current output** (`before`). Then split the jobs: *filter*, *add up*,
*format*. The category totals can reuse `spending_by_category` from section 3.

In [ ]:
def transactions_in_month(transactions, month):
    """Return only the transactions whose date starts with month (YYYY-MM)."""
    return [item for item in transactions if item["date"][:7] == month]


def total_of_kind(transactions, kind):
    """Return the sum of the amounts of one kind ("income" or "expense")."""
    return sum([item["amount"] for item in transactions if item["kind"] == kind])


def monthly_report(transactions, month):
    """Return a text report for one month (YYYY-MM)."""
    in_month = transactions_in_month(transactions, month)
    income = total_of_kind(in_month, "income")
    spent = total_of_kind(in_month, "expense")
    lines = [f"Report for {month}", f"Income: {income:.2f}", f"Spent:  {spent:.2f}"]
    for category, total in spending_by_category(in_month).items():
        lines.append(f"  {category:<10}{total:>8.2f}")
    lines.append(f"Balance: {income - spent:.2f}")
    return "\n".join(lines)


assert monthly_report(history, "2026-12") == before
print("same output as before: refactoring is safe")

Each new function has one job, a name that says it, and can be tested alone. The `assert ... == before` proves we
changed the structure, not the behaviour.

## 5. Adding input validation

Two layers, as in Projects 03–04: a **parse** function that checks text and raises `ValueError` (easy to test), and
an **ask** loop that keeps asking (only used by the menu).

In [ ]:
def parse_amount(text):
    """Turn text such as ' 12.50 ' into a positive float. Raises ValueError with a friendly message."""
    text = text.strip()
    if text == "":
        raise ValueError("Please type an amount.")
    try:
        amount = float(text)
    except ValueError:
        raise ValueError(f"'{text}' is not a number.")
    if amount <= 0:
        raise ValueError("The amount must be more than 0.")
    return round(amount, 2)


assert parse_amount(" 12.50 ") == 12.5
for bad in ["", "twelve", "-5", "0"]:
    try:
        parse_amount(bad)
        print("NOT rejected:", bad)
    except ValueError as error:
        print(f"'{bad}' -> {error}")

In [ ]:
def ask_amount(prompt):
    """Ask until the user types a valid amount; return it."""
    while True:
        try:
            return parse_amount(input(prompt))
        except ValueError as error:
            print(f"  {error} Try again.")

🤔 **Predict:** the user types `twelve`, then `-5`, then nothing, then `12.50`. What is printed?

In [ ]:
amount = ask_amount("Amount: ")
print("Saved amount:", amount)

## 6. A code review checklist

Use it on your own code and, at mid-session today, on a classmate's.

| # | Check |
|---|-------|
| 1 | `uv run python main.py` and `uv run pytest -q` both work from a fresh terminal |
| 2 | Names say what things are (`expenses`, `total_spent`, not `l`, `t`, `x2`) |
| 3 | Every function has a docstring and one job; nothing longer than ~20 lines |
| 4 | Logic modules **return** values; only `main.py` calls `input()` / prints the menu |
| 5 | Every `input()` goes through a parse/ask helper. Try `abc`, an empty answer, `-1` |
| 6 | Data: a missing file → fresh start; a restart keeps the data |
| 7 | Class methods refuse invalid values with `raise ValueError` |
| 8 | Tests check exact values, including an edge case and an error case |
| 9 | No leftover debug prints, commented-out code or unused imports |

🤔 How many checklist items does this function break?

In [ ]:
def f(l, c):
    t = 0
    for x in l:
        if x["category"] == c:
            t = t + x["amount"]
    print(t)


f(sample, "food")

At least 4: names (#2), no docstring (#3), prints instead of returning (#4), and it adds income too if a category
name is shared (a hidden bug, so a test is missing, #8). Reviewed version:

In [ ]:
def spent_in_category(transactions, category):
    """Return the total spent (expenses only) in one category."""
    return spending_by_category(transactions).get(category, 0)


assert spent_in_category(sample, "food") == 80
assert spent_in_category(sample, "travel") == 0
print(spent_in_category(sample, "food"))

## Recap

1. Build in **small steps**: write → run → test → checkpoint. The app always runs.
2. Bugs that do not crash are found by **tests that know the answer** (tiny data you can check in your head).
3. Read tracebacks bottom-up, then ask which *value* caused the error, and decide what the code should do.
4. **Refactor** by saving the old output, splitting jobs into functions, and asserting the output is unchanged.
5. Validate input in two layers: `parse_…` raises `ValueError`; `ask_…` loops until valid.

**Next time:** your 5-minute demo and the Python readiness assessment.